# 04 — Random Forest (FAST + SAFE) + Recall Optimization (Artifacts → `models/`)

This version avoids long runtimes by:
- sampling the training set (keep all class 2 & 1, sample class 0),
- using `OrdinalEncoder` for `HOOD_158_CODE` (no huge one-hot),
- bounding RF depth and using limited CPU threads.

**Outputs (to `models/`):**
- `random_forest.joblib`
- `random_forest_tau.txt`


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, recall_score, precision_score


BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"

OUT_RF = MODEL_DIR / "random_forest.joblib"
OUT_TAU = MODEL_DIR / "random_forest_tau.txt"

In [2]:
df = pd.read_csv(SUPERVISED_PATH, low_memory=False)
df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)
df["y_class"] = pd.to_numeric(df["y_class"], errors="coerce").astype("int8")
df = df.sort_values(["time_3h","HOOD_158_CODE"]).reset_index(drop=True)

t = df["time_3h"]
train_end = pd.Timestamp("2024-12-31 23:59:59")
val_end   = pd.Timestamp("2025-06-30 23:59:59")

train_mask = t <= train_end
val_mask   = (t > train_end) & (t <= val_end)
test_mask  = t > val_end

y = df["y_class"].copy()
X = df.drop(columns=[c for c in ["y_class","y_count_next"] if c in df.columns], errors="ignore").copy()
X = X.drop(columns=["time_3h"], errors="ignore")

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val     = X.loc[val_mask], y.loc[val_mask]
X_test, y_test   = X.loc[test_mask], y.loc[test_mask]

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Train class dist:", y_train.value_counts().to_dict())


Train/Val/Test: (922720, 47) (228784, 47) (232418, 47)
Train class dist: {0: 823183, 1: 86350, 2: 13187}


In [3]:
def stratified_sample(X, y, frac0=0.06, max_rows=250_000, seed=42):
    idx0 = y[y==0].index
    idx1 = y[y==1].index
    idx2 = y[y==2].index

    s0 = idx0.to_series().sample(frac=frac0, random_state=seed).index if len(idx0) else idx0
    idx = s0.union(idx1).union(idx2)

    if max_rows is not None and len(idx) > max_rows:
        idx = pd.Index(idx).to_series().sample(n=max_rows, random_state=seed).index

    return X.loc[idx].copy(), y.loc[idx].copy()

# Sample TRAIN only (keep minorities fully, downsample class 0 moderately)
Xtr_s, ytr_s = stratified_sample(X_train, y_train, frac0=0.25, max_rows=600_000, seed=42)

# DO NOT sample validation (threshold tuning must reflect reality)
Xva_s, yva_s = X_val.copy(), y_val.copy()

print("Sampled train:", Xtr_s.shape, ytr_s.value_counts(normalize=True).round(4).to_dict())
print("Full val:", Xva_s.shape, yva_s.value_counts(normalize=True).round(4).to_dict())


Sampled train: (305333, 47) {0: 0.674, 1: 0.2828, 2: 0.0432}
Full val: (228784, 47) {0: 0.8951, 1: 0.0907, 2: 0.0141}


In [4]:
cat_cols = [c for c in Xtr_s.columns if Xtr_s[c].dtype == "object"]
if "HOOD_158_CODE" in Xtr_s.columns and "HOOD_158_CODE" not in cat_cols:
    cat_cols.append("HOOD_158_CODE")
num_cols = [c for c in Xtr_s.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), cat_cols),
    ],
    remainder="drop"
)

cpu = os.cpu_count() or 4
N_JOBS = max(1, cpu // 2)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=20,
    max_features="sqrt",
    bootstrap=True,
    max_samples=0.5,              # speeds up & regularizes (per-tree subsample)
    n_jobs=N_JOBS,
    random_state=42,
    class_weight="balanced_subsample"
)

rf_pipe = Pipeline([("preprocess", preprocess), ("model", rf)])
print("Training RF (n_jobs =", N_JOBS, ") ...")
rf_pipe.fit(Xtr_s, ytr_s)


Training RF (n_jobs = 4 ) ...


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['collisions',
                                                   'injury_collisions',
                                                   'ftr_collisions',
                                                   'pd_collisions',
                                                   'pedestrian_collisions',
                                                   'bicycle_collisions',
                                                   'pressure_sea', 'wind_speed',
                                                   'relative_humidity',
                                                   'temperature',
                                                   'cloud_cover_8', 'rain',
                                                   'snow', 'visibility',
                                                   '...
                                                   'ftr_collisions_lag_1', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ord',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['HOOD_158_CODE'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_samples=0.5, min_samples_leaf=20,
                                        n_estimators=500, n_jobs=4,
                                        random_state=42))])

In [5]:
def apply_two_thresholds(proba, tau1, tau2):
    """
    Decision rule:
      if P2 >= tau2 -> 2
      elif P1 >= tau1 -> 1
      else -> 0
    """
    p1 = proba[:, 1]
    p2 = proba[:, 2]
    pred = np.zeros(len(p1), dtype=int)
    pred[p1 >= tau1] = 1
    pred[p2 >= tau2] = 2  # overwrite -> class 2 wins
    return pred

def tune_two_thresholds(proba_val, y_val,
                        tau1_grid=np.linspace(0.20, 0.70, 11),
                        tau2_grid=np.linspace(0.05, 0.40, 15),
                        min_recall_1=0.25,
                        max_pred2_rate=0.15):
    rows = []
    for tau1 in tau1_grid:
        for tau2 in tau2_grid:
            pred = apply_two_thresholds(proba_val, tau1, tau2)
            rec = recall_score(y_val, pred, labels=[0,1,2], average=None, zero_division=0)
            macro = recall_score(y_val, pred, average="macro", zero_division=0)
            rec_high = recall_score((y_val==2).astype(int), (pred==2).astype(int), zero_division=0)
            prec_high = precision_score((y_val==2).astype(int), (pred==2).astype(int), zero_division=0)
            pred2_rate = float((pred==2).mean())
            rows.append((tau1, tau2, macro, rec[1], rec_high, prec_high, pred2_rate))
    df = pd.DataFrame(rows, columns=["tau1","tau2","macro_recall","recall_1","recall_2","precision_2","pred2_rate"])
    feasible = df[(df["recall_1"] >= min_recall_1) & (df["pred2_rate"] <= max_pred2_rate)]
    best = (feasible if len(feasible) else df).sort_values(
        ["macro_recall","recall_2","precision_2"], ascending=False
    ).iloc[0]
    return float(best["tau1"]), float(best["tau2"]), df


In [6]:
proba_val = rf_pipe.predict_proba(Xva_s)
tau1, tau2, grid = tune_two_thresholds(proba_val, yva_s, min_recall_1=0.25, max_pred2_rate=0.15)
print("Chosen thresholds:", {"tau1": tau1, "tau2": tau2})
display(grid.sort_values(["macro_recall","recall_2","precision_2"], ascending=False).head(10))

proba_test = rf_pipe.predict_proba(X_test)
pred_test_thr = apply_two_thresholds(proba_test, tau1, tau2)

print("=== RF Two-Threshold - Test ===")
print(classification_report(y_test, pred_test_thr, digits=4, zero_division=0))
print("Predicted class-2 rate:", float((pred_test_thr==2).mean()))


Chosen thresholds: {'tau1': 0.4, 'tau2': 0.3}


,tau1,tau2,macro_recall,recall_1,recall_2,precision_2,pred2_rate
69,0.40,0.275,0.500850,0.351383,0.563833,0.045624,0.174746
68,0.40,0.250,0.499572,0.284999,0.635240,0.040026,0.224413
67,0.40,0.225,0.497432,0.216639,0.709119,0.035411,0.283158
70,0.40,0.300,0.497345,0.407024,0.491190,0.051884,0.133864
54,0.35,0.275,0.496727,0.495905,0.563833,0.045624,0.174746
71,0.40,0.325,0.495132,0.456258,0.429366,0.060547,0.100274
53,0.35,0.250,0.494658,0.419453,0.635240,0.040026,0.224413
55,0.35,0.300,0.494221,0.560410,0.491190,0.051884,0.133864
56,0.35,0.325,0.493454,0.619472,0.429366,0.060547,0.100274
66,0.40,0.200,0.492671,0.155025,0.774961,0.031431,0.348635


=== RF Two-Threshold - Test ===
              precision    recall  f1-score   support

           0     0.9505    0.5756    0.7170    207818
           1     0.1190    0.4103    0.1845     21189
           2     0.0537    0.5280    0.0975      3411

    accuracy                         0.5598    232418
   macro avg     0.3744    0.5046    0.3330    232418
weighted avg     0.8615    0.5598    0.6593    232418

Predicted class-2 rate: 0.1443347761360996


In [8]:
proba_val = rf_pipe.predict_proba(Xva_s)
tau1, tau2, grid = tune_two_thresholds(proba_val, yva_s, min_recall_1=0.25, max_pred2_rate=0.15)
print("Chosen thresholds:", {"tau1": tau1, "tau2": tau2})
display(grid.sort_values(["macro_recall","recall_2","precision_2"], ascending=False).head(10))

proba_test = rf_pipe.predict_proba(X_test)
pred_test_thr = apply_two_thresholds(proba_test, tau1, tau2)

print("=== RF Two-Threshold - Test ===")
print(classification_report(y_test, pred_test_thr, digits=4, zero_division=0))
print("Predicted class-2 rate:", float((pred_test_thr==2).mean()))

Chosen thresholds: {'tau1': 0.4, 'tau2': 0.3}


,tau1,tau2,macro_recall,recall_1,recall_2,precision_2,pred2_rate
69,0.40,0.275,0.500850,0.351383,0.563833,0.045624,0.174746
68,0.40,0.250,0.499572,0.284999,0.635240,0.040026,0.224413
67,0.40,0.225,0.497432,0.216639,0.709119,0.035411,0.283158
70,0.40,0.300,0.497345,0.407024,0.491190,0.051884,0.133864
54,0.35,0.275,0.496727,0.495905,0.563833,0.045624,0.174746
71,0.40,0.325,0.495132,0.456258,0.429366,0.060547,0.100274
53,0.35,0.250,0.494658,0.419453,0.635240,0.040026,0.224413
55,0.35,0.300,0.494221,0.560410,0.491190,0.051884,0.133864
56,0.35,0.325,0.493454,0.619472,0.429366,0.060547,0.100274
66,0.40,0.200,0.492671,0.155025,0.774961,0.031431,0.348635


=== RF Two-Threshold - Test ===
              precision    recall  f1-score   support

           0     0.9505    0.5756    0.7170    207818
           1     0.1190    0.4103    0.1845     21189
           2     0.0537    0.5280    0.0975      3411

    accuracy                         0.5598    232418
   macro avg     0.3744    0.5046    0.3330    232418
weighted avg     0.8615    0.5598    0.6593    232418

Predicted class-2 rate: 0.1443347761360996
